# Campsite Recon — Raw API Inspector

Debug tool for inspecting raw JSON responses from Recreation.gov and Open-Meteo.
Use this to verify facility IDs, understand response shapes, and diagnose issues in the production code (`recon/`).

**Two ways to use this notebook:**
- **Interactive UI** — run all cells, then use the widget panel at the bottom
- **Manual** — edit the Variables cell and run individual sections

In [ ]:
import json, os, subprocess, ssl, certifi
import ipywidgets as widgets
from IPython.display import display, clear_output
from datetime import date, timedelta
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

# ── API key from Keychain ────────────────────────────────────────────────────
def _get_api_key():
    user = os.environ.get('USER', '')
    try:
        return subprocess.check_output(
            ['security', 'find-generic-password', '-a', user, '-s', 'recreation-gov-api', '-w'],
            text=True, stderr=subprocess.DEVNULL
        ).strip()
    except Exception:
        return os.environ.get('RIDB_API_KEY', '')

RIDB_API_KEY = _get_api_key()
assert RIDB_API_KEY, 'No API key — store it in Keychain under recreation-gov-api'

_SSL     = ssl.create_default_context(cafile=certifi.where())
_HEADERS = {'User-Agent': 'CampsiteRecon/1.0', 'Accept': 'application/json'}

def get(url):
    """GET request — prints status, returns parsed JSON or None on failure."""
    try:
        with urlopen(Request(url, headers=_HEADERS), context=_SSL, timeout=15) as r:
            print(f'✅ {r.status}  {url}')
            return json.loads(r.read().decode('utf-8'))
    except HTTPError as e:
        print(f'❌ HTTP {e.code}  {e.read().decode()[:300]}')
    except URLError as e:
        print(f'❌ Network: {e.reason}')
    return None

def pp(data, limit=60):
    """Pretty-print JSON, capped at `limit` lines."""
    lines = json.dumps(data, indent=2, default=str).splitlines()
    for line in lines[:limit]:
        print(line)
    if len(lines) > limit:
        print(f'... +{len(lines) - limit} lines hidden — raise limit= to see more')

def resolve_friday(target=None):
    if target:
        return date.fromisoformat(target)
    today = date.today()
    days  = (4 - today.weekday()) % 7 or 7
    return today + timedelta(days=days)

print('✅ ready')

---
## Variables
Change these to inspect specific facilities or dates manually. The interactive panel below also sets these.

In [ ]:
# ── Facility / permit to inspect ─────────────────────────────────────────────
FACILITY_ID = '234059'    # Sky Camp (Point Reyes)
PERMIT_ID   = '4675310'   # Sky Camp (Point Reyes)

# Known IDs for reference:
#   Point Reyes  — Sky Camp:     facility 234059  permit 4675310
#                  Coast Camp:   facility 234061  permit 4675311
#                  Glen Camp:    facility 234060  permit 4675312
#                  Wildcat Camp: facility 234062  permit 4675313
#   Big Sur      — Kirk Creek:       facility 233116  (no permit)
#                  Plaskett Creek:   facility 233115  (no permit)
#                  Pfeiffer Big Sur: facility 233394  (no permit)
#                  Andrew Molera:    facility 234218  (no permit)

TARGET_FRIDAY = None      # YYYY-MM-DD string, or None for next Friday
SEARCH_QUERY  = 'point reyes'
LAT, LON      = 38.037, -122.803   # Point Reyes coords for weather

# ── Resolved (don't edit) ────────────────────────────────────────────────────
FRIDAY      = resolve_friday(TARGET_FRIDAY)
SATURDAY    = FRIDAY + timedelta(1)
SUNDAY      = FRIDAY + timedelta(2)
MONTH_PARAM = f'{FRIDAY.year}-{FRIDAY.month:02d}-01T00%3A00%3A00.000Z'

print(f'Weekend  : {FRIDAY} (Fri)  {SATURDAY} (Sat)  {SUNDAY} (Sun)')
print(f'Month    : {MONTH_PARAM}')
print(f'Facility : {FACILITY_ID}   Permit: {PERMIT_ID}')

---
## RIDB — Facility Search

Use this to discover facility IDs for new locations.
RIDB is the official catalogue of recreation facilities — separate from the availability API.

### Key fields in each result (`RECDATA` array)

| Field | Type | Notes |
|---|---|---|
| `FacilityID` | string | **The ID you need for the availability endpoint.** Permanent — does not change. |
| `FacilityName` | string | Human-readable name, often uppercase. |
| `FacilityTypeDescription` | string | Filter on `"Campground"` — otherwise you get visitor centres, parking areas, etc. |
| `FacilityLatitude` / `FacilityLongitude` | float | Coordinates for weather calls. Can be `null` for some facilities. |
| `Reservable` | bool | `true` = can be booked online. Visitor centres are always `false`. |
| `RECAREA` | array | Parent rec area — confirms you have the right park. |
| `METADATA.RESULTS.TOTAL_COUNT` | int | Total matches (you may only receive the first `limit`). |

**When search returns nothing:** try `/recareas?query={name}` → get `RecAreaID` → then `/recareas/{id}/facilities?facilitytype=Campground`

In [ ]:
ridb_url = (
    f'https://ridb.recreation.gov/api/v1/facilities'
    f'?query={SEARCH_QUERY.replace(" ", "+")}'
    f'&facilitytype=Campground&radius=30&limit=10&apikey={RIDB_API_KEY}'
)
ridb_raw = get(ridb_url)

if ridb_raw:
    total = ridb_raw.get('METADATA', {}).get('RESULTS', {}).get('TOTAL_COUNT', '?')
    print(f'\nTotal in database: {total}\n')
    pp(ridb_raw)

In [ ]:
# Condensed view — just the fields we care about
if ridb_raw:
    print(f'{"FacilityID":<14} {"Reservable":<12} {"Lat":>8} {"Lon":>10}  Name')
    print('─' * 80)
    for f in ridb_raw.get('RECDATA', []):
        print(
            f'{f.get("FacilityID","?"):<14}'
            f' {str(f.get("Reservable","?")):<12}'
            f' {str(f.get("FacilityLatitude") or "—"):>8}'
            f' {str(f.get("FacilityLongitude") or "—"):>10}'
            f'  {f.get("FacilityName","?")}'
        )

---
## Campground Availability Endpoint

Standard endpoint for drive-in campgrounds (Big Sur, most national forest sites).

**URL:** `GET /api/camps/availability/campground/{facilityId}/month?start_date={YYYY-MM-01T00%3A00%3A00.000Z}`

> ⚠️ **Colons must be URL-encoded as `%3A`.** Raw colons return `400 {"error":"query not encoded"}` — silently
> caught as `None` by the HTTP client. Discovered during smoke test April 2026.

### Top-level response structure
```json
{
  "campsites": {
    "<internal_site_id>": {
      "site": "A1",
      "loop": "Main Loop",
      "campsite_reserve_type": "Site-Specific",
      "quantities": null,
      "availabilities": {
        "2026-04-17T00:00:00Z": "Available",
        "2026-04-18T00:00:00Z": "Reserved"
      }
    }
  }
}
```
If `campsites` is absent or empty: facility uses the permit endpoint instead (e.g. Point Reyes).

### Field notes

| Field | Notes |
|---|---|
| `campsites` (top-level) | Dict keyed by internal site ID — one entry per physical campsite. |
| `site` | Human label, e.g. `"A1"`. Useful for displaying specific site names. |
| `loop` | Loop name within the campground. |
| `campsite_reserve_type` | Usually `"Site-Specific"` for individual reservable sites. |
| `quantities` | Group site capacity. Usually `null` for individual sites. |
| `availabilities` | **Key field.** ISO timestamp → status. Use `dt_str[:10]` to get date only. |

### Status values in `availabilities`

| Status | Bookable |
|---|---|
| `Available` | ✅ |
| `Open` | ✅ (walk-up / first-come) |
| `Reserved` | ❌ |
| `Not Available` | ❌ (closed or not offered) |
| `Not Reservable` | ❌ (walk-up only, no online booking) |
| `Not Reservable Management` | ❌ (held by park staff) |

A campground can return 30+ sites — always aggregate across all of them to count open slots per date.

In [ ]:
cg_url = (
    f'https://www.recreation.gov/api/camps/availability/campground'
    f'/{FACILITY_ID}/month?start_date={MONTH_PARAM}'
)
cg_raw = get(cg_url)

if cg_raw:
    sites = cg_raw.get('campsites', {})
    print(f'\nTotal sites returned: {len(sites)}')
    if not sites:
        print('campsites is empty — try the permit endpoint for this facility')
    pp(cg_raw, limit=60)

In [ ]:
# Weekend summary — aggregated across all sites
if cg_raw and cg_raw.get('campsites'):
    targets = {FRIDAY, SATURDAY, SUNDAY}
    summary = {}

    for site in cg_raw['campsites'].values():
        for dt_str, status in site.get('availabilities', {}).items():
            try:
                d = date.fromisoformat(dt_str[:10])
            except ValueError:
                continue
            if d not in targets:
                continue
            iso = d.isoformat()
            entry = summary.setdefault(iso, {'available': 0, 'total': 0, 'statuses': set()})
            entry['total'] += 1
            entry['statuses'].add(status)
            if status in ('Available', 'Open'):
                entry['available'] += 1

    print(f'{"Date":<14} {"Open":>6} {"Total":>6}   All statuses seen')
    print('─' * 65)
    for dt in sorted(summary):
        a    = summary[dt]['available']
        t    = summary[dt]['total']
        s    = ', '.join(sorted(summary[dt]['statuses']))
        icon = '✅' if a > 0 else '❌'
        print(f'{dt:<14} {a:>6} {t:>6}   {icon} {s}')

---
## Permit Availability Endpoint

Wilderness / backcountry permit campgrounds (Point Reyes, etc.). Entirely different response shape.

**URL:** `GET /api/permits/{permitId}/availability/month?start_date={YYYY-MM-01T00%3A00%3A00.000Z}&commercial_acct=false`

> Always include `&commercial_acct=false` — omitting it may return commercial quotas instead of public slots.

### Top-level response structure
```json
{
  "payload": {
    "availability": {
      "2026-04-17T00:00:00Z": { "remaining": 2, "total": 8, "status": "Available" },
      "2026-04-18T00:00:00Z": { "remaining": 0, "total": 8, "status": "Sold Out" }
    }
  }
}
```
> Some responses omit `payload` entirely. Always access via `raw.get('payload', raw).get('availability', {})`.

### Field notes

| Field | Notes |
|---|---|
| `remaining` | Permit slots left for this entry date. **The only field that matters.** Can be `null` — not just 0. |
| `total` | Daily quota. Useful for showing `remaining / total` context to the user. |
| `status` | Human label (`"Available"`, `"Sold Out"`). Less reliable than checking `remaining > 0` directly. |

Always check `isinstance(remaining, int) and remaining > 0`.

**Point Reyes note:** Fully booked is the norm. 90-day window opens at midnight and clears in minutes.
Availability = last-minute cancellations only.

In [ ]:
permit_url = (
    f'https://www.recreation.gov/api/permits/{PERMIT_ID}/availability/month'
    f'?start_date={MONTH_PARAM}&commercial_acct=false'
)
permit_raw = get(permit_url)

if permit_raw:
    print(f'\nTop-level keys: {list(permit_raw.keys())}')
    payload = permit_raw.get('payload', permit_raw)
    avail   = payload.get('availability', {})
    print(f'Dates in availability: {len(avail)}')
    pp(permit_raw, limit=60)

In [ ]:
# Weekend summary
if permit_raw:
    payload = permit_raw.get('payload', permit_raw)
    avail   = payload.get('availability', {})
    targets = {FRIDAY, SATURDAY, SUNDAY}

    print(f'{"Date":<14} {"Remaining":>10} {"Total":>7}   Status')
    print('─' * 55)
    found = False

    for dt_str, info in sorted(avail.items()):
        try:
            d = date.fromisoformat(dt_str[:10])
        except ValueError:
            continue
        if d not in targets:
            continue
        found    = True
        rem      = info.get('remaining')
        icon     = '✅' if isinstance(rem, int) and rem > 0 else '❌'
        rem_disp = str(rem) if rem is not None else 'null'
        print(f'{d.isoformat():<14} {rem_disp:>10} {str(info.get("total","?")):>7}   {icon} {info.get("status","?")}')

    if not found:
        print('No data for target weekend dates')

---
## Open-Meteo — Weather

Free, no key required. 14-day daily forecast. All `daily` fields are **parallel arrays** — index `i` in
`time` matches index `i` in every other field.

**URL:** `GET https://api.open-meteo.com/v1/forecast?latitude=…&longitude=…&daily=…&timezone=auto`

> Always include `&timezone=auto` — without it, dates are UTC and misalign with local Pacific time.

### Field notes

| Field | Notes |
|---|---|
| `time` | `YYYY-MM-DD` — no timestamp, no parsing. Direct string comparison with `date.isoformat()` works. |
| `weathercode` | WMO code — see table below. |
| `temperature_2m_max` / `temperature_2m_min` | Daily high/low in requested unit (we use Celsius). |
| `precipitation_sum` | mm of rain. **Can be `null`** — always coerce with `value or 0.0`. |
| `windspeed_10m_max` | km/h. Flag > 25 kph for exposed coastal or ridge campsites. |
| `timezone` / `timezone_abbreviation` | Top-level response fields — confirms local time resolved correctly. |

### WMO codes

| Code | Condition | Code | Condition |
|---|---|---|---|
| 0 | Clear | 61–65 | Rain |
| 1 | Mainly clear | 71–75 | Snow |
| 2 | Partly cloudy | 77 | Snow grains |
| 3 | Overcast | 80–82 | Showers |
| 45, 48 | Fog | 95 | Thunderstorm |
| 51–55 | Drizzle | 96, 99 | Thunderstorm + hail |

In [ ]:
wx_url = (
    f'https://api.open-meteo.com/v1/forecast'
    f'?latitude={LAT}&longitude={LON}'
    f'&daily=weathercode,temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max'
    f'&temperature_unit=celsius&wind_speed_unit=kmh&precipitation_unit=mm'
    f'&timezone=auto&forecast_days=14'
)
wx_raw = get(wx_url)

if wx_raw:
    print(f'\nTimezone : {wx_raw.get("timezone","?")} ({wx_raw.get("timezone_abbreviation","?")})')
    print(f'Days     : {len(wx_raw.get("daily",{}).get("time",[]))}')
    pp(wx_raw)

In [ ]:
WMO = {
    0: 'Clear', 1: 'Mainly clear', 2: 'Partly cloudy', 3: 'Overcast',
    45: 'Fog', 48: 'Icy fog',
    51: 'Light drizzle', 53: 'Drizzle', 55: 'Heavy drizzle',
    61: 'Light rain', 63: 'Rain', 65: 'Heavy rain',
    71: 'Light snow', 73: 'Snow', 75: 'Heavy snow', 77: 'Snow grains',
    80: 'Showers', 81: 'Showers', 82: 'Heavy showers',
    95: 'Thunderstorm', 96: 'Thunderstorm + hail', 99: 'Thunderstorm + hail',
}

if wx_raw:
    daily   = wx_raw.get('daily', {})
    times   = daily.get('time', [])
    targets = {FRIDAY.isoformat(), SATURDAY.isoformat(), SUNDAY.isoformat()}

    print(f'{"Date":<14} {"High":>7} {"Low":>7} {"Rain":>8} {"Wind":>9}   Condition')
    print('─' * 72)
    for i, t in enumerate(times):
        if t not in targets:
            continue
        rain = daily['precipitation_sum'][i] or 0.0
        wind = daily['windspeed_10m_max'][i]
        flag = '  ⚠️ high wind' if wind > 25 else ''
        print(
            f'{t:<14}'
            f' {daily["temperature_2m_max"][i]:>6.1f}C'
            f' {daily["temperature_2m_min"][i]:>6.1f}C'
            f' {rain:>6.1f}mm'
            f' {wind:>7.1f}kph'
            f'   {WMO.get(daily["weathercode"][i], "WMO " + str(daily["weathercode"][i]))}'
            f'{flag}'
        )

---
## Interactive Inspector
Select a location, pick what to inspect, and hit **Run**. Updates the variables above so individual cells can be re-run against the same selection.

In [ ]:
# ── Known locations ───────────────────────────────────────────────────────────
KNOWN = {
    'Point Reyes — Sky Camp':     {'facility': '234059', 'permit': '4675310', 'lat': 38.037, 'lon': -122.803},
    'Point Reyes — Coast Camp':   {'facility': '234061', 'permit': '4675311', 'lat': 38.017, 'lon': -122.924},
    'Point Reyes — Glen Camp':    {'facility': '234060', 'permit': '4675312', 'lat': 38.001, 'lon': -122.875},
    'Point Reyes — Wildcat Camp': {'facility': '234062', 'permit': '4675313', 'lat': 37.965, 'lon': -122.862},
    'Big Sur — Kirk Creek':       {'facility': '233116', 'permit': '',        'lat': 35.685, 'lon': -121.149},
    'Big Sur — Plaskett Creek':   {'facility': '233115', 'permit': '',        'lat': 35.672, 'lon': -121.145},
    'Big Sur — Pfeiffer':         {'facility': '233394', 'permit': '',        'lat': 36.250, 'lon': -121.782},
    'Big Sur — Andrew Molera':    {'facility': '234218', 'permit': '',        'lat': 36.079, 'lon': -121.846},
    'Custom':                     {'facility': '',        'permit': '',        'lat': None,   'lon': None},
}

# ── Widgets ───────────────────────────────────────────────────────────────────
style  = {'description_width': '100px'}
layout = widgets.Layout(width='400px')

loc_w  = widgets.Dropdown(
    options=list(KNOWN.keys()), description='📍 Location:',
    style=style, layout=layout
)
fac_w  = widgets.Text(description='Facility ID:', style=style, layout=layout)
per_w  = widgets.Text(description='Permit ID:',   style=style, layout=layout)
fri_w  = widgets.Text(
    description='Friday date:',
    placeholder='YYYY-MM-DD or blank for next Friday',
    style=style, layout=layout
)

chk_ridb  = widgets.Checkbox(value=False, description='RIDB facility lookup')
chk_cg    = widgets.Checkbox(value=True,  description='Campground availability')
chk_perm  = widgets.Checkbox(value=True,  description='Permit availability')
chk_wx    = widgets.Checkbox(value=True,  description='Weather')

btn = widgets.Button(
    description='Run Inspection',
    button_style='primary',
    layout=widgets.Layout(width='160px', margin='10px 0 0 0')
)
out = widgets.Output()

def on_location_change(change):
    loc = KNOWN[change['new']]
    fac_w.value = loc['facility']
    per_w.value = loc['permit']

loc_w.observe(on_location_change, names='value')
on_location_change({'new': loc_w.value})   # seed initial values

def on_run(b):
    global FACILITY_ID, PERMIT_ID, FRIDAY, SATURDAY, SUNDAY, MONTH_PARAM, LAT, LON

    loc        = KNOWN[loc_w.value]
    FACILITY_ID = fac_w.value.strip()
    PERMIT_ID   = per_w.value.strip()
    LAT         = loc['lat']
    LON         = loc['lon']
    FRIDAY      = resolve_friday(fri_w.value.strip() or None)
    SATURDAY    = FRIDAY + timedelta(1)
    SUNDAY      = FRIDAY + timedelta(2)
    MONTH_PARAM = f'{FRIDAY.year}-{FRIDAY.month:02d}-01T00%3A00%3A00.000Z'

    with out:
        clear_output(wait=True)
        print(f'Location : {loc_w.value}')
        print(f'Weekend  : {FRIDAY} (Fri)  {SATURDAY} (Sat)  {SUNDAY} (Sun)')
        print(f'Facility : {FACILITY_ID}   Permit: {PERMIT_ID}')
        print('─' * 70)

        if chk_ridb.value and FACILITY_ID:
            print('\n── RIDB lookup ──')
            ridb_url = (
                f'https://ridb.recreation.gov/api/v1/facilities/{FACILITY_ID}'
                f'?apikey={RIDB_API_KEY}'
            )
            raw = get(ridb_url)
            if raw:
                pp(raw, limit=30)

        if chk_cg.value and FACILITY_ID:
            print('\n── Campground availability ──')
            cg_url = (
                f'https://www.recreation.gov/api/camps/availability/campground'
                f'/{FACILITY_ID}/month?start_date={MONTH_PARAM}'
            )
            raw = get(cg_url)
            if raw:
                sites = raw.get('campsites', {})
                print(f'Sites returned: {len(sites)}')
                if not sites:
                    print('Empty — try permit endpoint instead')
                else:
                    targets = {FRIDAY, SATURDAY, SUNDAY}
                    summary = {}
                    for site in sites.values():
                        for dt_str, status in site.get('availabilities', {}).items():
                            try:
                                d = date.fromisoformat(dt_str[:10])
                            except ValueError:
                                continue
                            if d not in targets:
                                continue
                            iso = d.isoformat()
                            entry = summary.setdefault(iso, {'available': 0, 'total': 0})
                            entry['total'] += 1
                            if status in ('Available', 'Open'):
                                entry['available'] += 1
                    for dt in sorted(summary):
                        a    = summary[dt]['available']
                        t    = summary[dt]['total']
                        icon = '✅' if a > 0 else '❌'
                        print(f'  {dt}  {icon}  {a}/{t} sites open')

        if chk_perm.value and PERMIT_ID:
            print('\n── Permit availability ──')
            permit_url = (
                f'https://www.recreation.gov/api/permits/{PERMIT_ID}/availability/month'
                f'?start_date={MONTH_PARAM}&commercial_acct=false'
            )
            raw = get(permit_url)
            if raw:
                payload = raw.get('payload', raw)
                avail   = payload.get('availability', {})
                targets = {FRIDAY, SATURDAY, SUNDAY}
                found   = False
                for dt_str, info in sorted(avail.items()):
                    try:
                        d = date.fromisoformat(dt_str[:10])
                    except ValueError:
                        continue
                    if d not in targets:
                        continue
                    found = True
                    rem   = info.get('remaining')
                    icon  = '✅' if isinstance(rem, int) and rem > 0 else '❌'
                    print(f'  {d.isoformat()}  {icon}  {rem}/{info.get("total","?")} permits  ({info.get("status","?")})')
                if not found:
                    print('  No data for target dates')

        if chk_wx.value and LAT and LON:
            print('\n── Weather ──')
            wx_url = (
                f'https://api.open-meteo.com/v1/forecast'
                f'?latitude={LAT}&longitude={LON}'
                f'&daily=weathercode,temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max'
                f'&temperature_unit=celsius&wind_speed_unit=kmh&precipitation_unit=mm'
                f'&timezone=auto&forecast_days=14'
            )
            raw = get(wx_url)
            if raw:
                daily   = raw.get('daily', {})
                times   = daily.get('time', [])
                targets = {FRIDAY.isoformat(), SATURDAY.isoformat(), SUNDAY.isoformat()}
                for i, t in enumerate(times):
                    if t not in targets:
                        continue
                    rain = daily['precipitation_sum'][i] or 0.0
                    wind = daily['windspeed_10m_max'][i]
                    flag = ' ⚠️' if wind > 25 else ''
                    print(
                        f'  {t}  '
                        f'{daily["temperature_2m_max"][i]:.0f}C / {daily["temperature_2m_min"][i]:.0f}C  '
                        f'rain {rain:.1f}mm  wind {wind:.0f}kph'
                        f'{flag}'
                    )

btn.on_click(on_run)

display(
    widgets.HTML('<b style="font-family:monospace;font-size:14px">Campsite Recon — Inspector</b>'),
    loc_w,
    widgets.HBox([fac_w]),
    widgets.HBox([per_w]),
    fri_w,
    widgets.HBox([chk_ridb, chk_cg, chk_perm, chk_wx]),
    btn,
    out
)